# Hierarchical Agent Delegation — Manager-Worker Agent Patterns

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/20_hierarchical_agent_delegation.ipynb)

## What This Notebook Teaches You

When a CEO receives a complex project, they don't do everything personally. They **decompose** the project into parts, **delegate** to department heads, who further delegate to specialists. The CEO then **aggregates** results into a coherent whole.

**Hierarchical agent systems** bring this same organizational pattern to LLM agents. A manager agent breaks complex tasks into subtasks and routes them to specialized workers.

**Research foundation**:
- Park et al. 2023, *"Generative Agents: Interactive Simulacra of Human Behavior"* — demonstrated emergent social hierarchies in agent populations
- Wu et al. 2023, *"AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation"* — formalized manager-worker patterns for multi-agent collaboration
- Hong et al. 2023, *"MetaGPT: Meta Programming for Multi-Agent Collaborative Framework"* — showed that role-based agent hierarchies produce better software than flat approaches

By the end of this notebook, you will:

1. **Understand hierarchical task decomposition** — why it outperforms flat approaches for complex tasks
2. **Build a WorkerAgent with typed capabilities** and a WorkerRegistry for discovery
3. **Build a ManagerAgent** that decomposes, delegates, and aggregates autonomously
4. **Implement both flat and hierarchical decomposition** strategies
5. **Handle worker failures** — retry, reassign, and graceful degradation
6. **Compare flat vs. manager-worker vs. multi-level hierarchy** approaches

---

> **Prerequisites**: Notebook 19 (Sequential Agent Pipelines), Notebooks 01–06 (fundamentals).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~60–75 minutes.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **What This Notebook Teaches You**
- Understand **: Environment Setup**
- Understand **Why Hierarchy?**
- Understand **Shared Data Structures**
- Understand **The WorkerAgent Class**


## Part 0: Environment Setup

Same Qwen3-8B model. In this notebook, one LLM instance plays multiple roles — manager, workers, sub-managers — differentiated entirely by system prompts.

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. Why Hierarchy?

### 1.1 — The Organizational Analogy

Every effective organization uses hierarchy for complex work:

```
        ┌──────────────┐
        │     CEO      │  ← Receives complex goal
        └──────┬───────┘
               │ decomposes into departments
    ┌──────────┼──────────┐
    ▼          ▼          ▼
┌───────┐ ┌───────┐ ┌───────┐
│  R&D  │ │ Sales │ │  Ops  │  ← Each has specialized expertise
└───┬───┘ └───┬───┘ └───┬───┘
    │         │         │
    ▼         ▼         ▼
 Workers   Workers   Workers    ← Execute specific subtasks
```

**Why this works:**
- **Cognitive limits**: No single person (or LLM call) can hold an entire complex project in working memory
- **Specialization**: Focused prompts produce better results than "do everything" prompts
- **Parallelism**: Independent subtasks can run simultaneously
- **Error containment**: A failure in one branch doesn't corrupt the whole project

### 1.2 — When Flat Approaches Fail

A single agent asked to "Write a comprehensive report on renewable energy" must simultaneously:
- Research solar, wind, hydro, nuclear, geothermal
- Analyze costs, efficiency, environmental impact for each
- Compare technologies
- Write clearly and organize logically

That's 5+ distinct tasks crammed into one prompt. The result: shallow coverage, missed topics, inconsistent quality.

### 1.3 — The Decompose-Delegate-Aggregate Pattern

```
            ┌──────────────────────┐
            │   DECOMPOSE          │ ← Break task into subtasks
            │   (Manager LLM call) │
            └──────────┬───────────┘
                       │
          ┌────────────┼────────────┐
          ▼            ▼            ▼
     ┌─────────┐ ┌─────────┐ ┌─────────┐
     │ DELEGATE │ │ DELEGATE │ │ DELEGATE │ ← Route to specialists
     │ Worker 1 │ │ Worker 2 │ │ Worker 3 │
     └────┬────┘ └────┬────┘ └────┬────┘
          │            │            │
          ▼            ▼            ▼
     ┌──────────────────────────────────┐
     │          AGGREGATE               │ ← Combine results
     │          (Manager LLM call)      │
     └──────────────────────────────────┘
```

## 2. Shared Data Structures

Before building workers and managers, we need common data types that both understand.

In [ ]:
from datetime import datetime
from enum import Enum

class TaskStatus(Enum):
    PENDING = "pending"
    IN_PROGRESS = "in_progress"
    COMPLETED = "completed"
    FAILED = "failed"
    REASSIGNED = "reassigned"

@dataclass
class SubTask:
    """A single subtask created by decomposition."""
    id: str
    description: str
    required_skills: List[str] = field(default_factory=list)
    status: TaskStatus = TaskStatus.PENDING
    assigned_to: Optional[str] = None
    result: Optional[str] = None
    error: Optional[str] = None
    attempts: int = 0

    def summary(self) -> str:
        status_icon = {
            TaskStatus.PENDING: "⏳",
            TaskStatus.IN_PROGRESS: "🔄",
            TaskStatus.COMPLETED: "✅",
            TaskStatus.FAILED: "❌",
            TaskStatus.REASSIGNED: "🔀",
        }
        icon = status_icon.get(self.status, "?")
        worker = f" → {self.assigned_to}" if self.assigned_to else ""
        return f"{icon} [{self.id}] {self.description[:60]}{worker}"

@dataclass
class TaskDecomposition:
    """Result of decomposing a complex task into subtasks."""
    original_task: str
    subtasks: List[SubTask]
    strategy: str = "flat"  # "flat" or "hierarchical"

    def show(self):
        print(f"Task: {self.original_task[:80]}...")
        print(f"Strategy: {self.strategy} | Subtasks: {len(self.subtasks)}")
        for st in self.subtasks:
            print(f"  {st.summary()}")

# Test
demo_decomp = TaskDecomposition(
    original_task="Write a report on renewable energy sources",
    subtasks=[
        SubTask("research-solar", "Research solar energy technology and costs", ["research", "energy"]),
        SubTask("research-wind", "Research wind energy technology and costs", ["research", "energy"]),
        SubTask("compare", "Compare all energy sources", ["analysis", "writing"]),
    ],
)
demo_decomp.show()

## 3. The WorkerAgent Class

A worker has a **specialty** (defined by skills and system prompt) and processes subtasks. Unlike pipeline stages, workers don't pass output to the next worker — they report back to the manager.

In [ ]:
@dataclass
class WorkerAgent:
    """A specialized worker agent with defined capabilities.

    Design decisions:
    - skills: list of capability tags for matching by the registry
    - system_prompt: defines the worker's expertise and behavior
    - max_tokens/temperature: tuned for the worker's task type
    """
    name: str
    skills: List[str]
    system_prompt: str
    max_tokens: int = 512
    temperature: float = 0.7

    def can_handle(self, required_skills: List[str]) -> float:
        """Score how well this worker matches the required skills (0.0 to 1.0)."""
        if not required_skills:
            return 0.5  # Generic match
        matched = sum(1 for s in required_skills if s in self.skills)
        return matched / len(required_skills)

    def execute(self, subtask: SubTask) -> SubTask:
        """Execute a subtask and return the updated subtask with results."""
        subtask.status = TaskStatus.IN_PROGRESS
        subtask.assigned_to = self.name
        subtask.attempts += 1

        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f"Task: {subtask.description}"},
        ]

        start = time.time()
        try:
            response = generate(
                messages,
                max_new_tokens=self.max_tokens,
                temperature=self.temperature,
            )
            subtask.result = response
            subtask.status = TaskStatus.COMPLETED
            elapsed = time.time() - start
            print(f"    ✓ {self.name} completed '{subtask.id}' ({elapsed:.1f}s)")
        except Exception as e:
            subtask.status = TaskStatus.FAILED
            subtask.error = str(e)
            print(f"    ✗ {self.name} failed on '{subtask.id}': {e}")

        return subtask

    def __repr__(self):
        return f"Worker('{self.name}', skills={self.skills})"

# Create some workers
research_worker = WorkerAgent(
    name="researcher",
    skills=["research", "information_gathering", "data_collection"],
    system_prompt="""You are a thorough research specialist. Your job:
1. Gather comprehensive information on the assigned topic
2. Include specific facts, statistics, and examples
3. Cover multiple perspectives and sources
4. Organize findings clearly with headers and bullet points
Be detailed and factual. Focus ONLY on the specific subtask assigned.""",
    max_tokens=600,
    temperature=0.7,
)

analysis_worker = WorkerAgent(
    name="analyst",
    skills=["analysis", "comparison", "evaluation", "critical_thinking"],
    system_prompt="""You are an analytical specialist. Your job:
1. Analyze data and findings provided or discovered
2. Identify patterns, trends, and key insights
3. Compare alternatives objectively
4. Provide evidence-based conclusions
Be rigorous and balanced. Support claims with reasoning.""",
    max_tokens=500,
    temperature=0.5,
)

writing_worker = WorkerAgent(
    name="writer",
    skills=["writing", "editing", "communication", "formatting"],
    system_prompt="""You are a professional writer. Your job:
1. Transform raw information into clear, engaging prose
2. Organize content with logical flow and structure
3. Use appropriate tone for the target audience
4. Ensure readability and conciseness
Produce polished, publication-ready text.""",
    max_tokens=600,
    temperature=0.6,
)

print(f"Workers created:")
for w in [research_worker, analysis_worker, writing_worker]:
    print(f"  {w}")
    print(f"    Skill match for ['research', 'energy']: {w.can_handle(['research', 'energy']):.0%}")
    print(f"    Skill match for ['writing', 'editing']: {w.can_handle(['writing', 'editing']):.0%}")

## 4. The WorkerRegistry — Discovering Workers by Skill

The manager shouldn't hardcode which worker handles what. Instead, a **registry** lets workers register their capabilities, and the manager queries "who can handle research tasks?" at runtime.

This is the same principle as **service discovery** in microservices architecture.

In [ ]:
class WorkerRegistry:
    """Registry for discovering workers by capability.

    Why a registry instead of direct references?
    1. Decoupling — manager doesn't need to know specific workers
    2. Extensibility — add new workers without changing manager code
    3. Load balancing — multiple workers can register the same skills
    4. Failover — if one worker fails, find another with matching skills
    """

    def __init__(self):
        self.workers: Dict[str, WorkerAgent] = {}

    def register(self, worker: WorkerAgent):
        """Register a worker agent."""
        self.workers[worker.name] = worker
        print(f"  📋 Registered: {worker.name} (skills: {', '.join(worker.skills)})")

    def find_worker(self, required_skills: List[str], exclude: List[str] = None) -> Optional[WorkerAgent]:
        """Find the best matching worker for the required skills.

        Args:
            required_skills: Skills needed for the task
            exclude: Worker names to skip (e.g., previously failed workers)
        """
        exclude = exclude or []
        candidates = [
            (worker, worker.can_handle(required_skills))
            for worker in self.workers.values()
            if worker.name not in exclude
        ]
        # Sort by match score (descending)
        candidates.sort(key=lambda x: x[1], reverse=True)

        if candidates and candidates[0][1] > 0:
            return candidates[0][0]
        return None

    def find_all_matching(self, required_skills: List[str], min_score: float = 0.3) -> List[WorkerAgent]:
        """Find all workers matching above the minimum score threshold."""
        return [
            worker for worker in self.workers.values()
            if worker.can_handle(required_skills) >= min_score
        ]

    def list_workers(self):
        """Print all registered workers and their skills."""
        print(f"\n📋 Worker Registry ({len(self.workers)} workers):")
        for name, worker in self.workers.items():
            print(f"  • {name}: {', '.join(worker.skills)}")

# Create and populate the registry
registry = WorkerRegistry()
print("Registering workers:")
registry.register(research_worker)
registry.register(analysis_worker)
registry.register(writing_worker)
registry.list_workers()

# Test skill matching
print("\nSkill matching tests:")
found = registry.find_worker(["research", "data_collection"])
print(f"  Need [research, data_collection] → {found.name if found else 'None'}")

found = registry.find_worker(["writing", "formatting"])
print(f"  Need [writing, formatting] → {found.name if found else 'None'}")

found = registry.find_worker(["analysis", "comparison"])
print(f"  Need [analysis, comparison] → {found.name if found else 'None'}")

## 5. The ManagerAgent — Decompose, Delegate, Aggregate

The manager is the brain of the hierarchy. It has three responsibilities:

1. **Decompose**: Break a complex task into subtasks (using LLM)
2. **Delegate**: Route each subtask to an appropriate worker (using registry)
3. **Aggregate**: Combine worker results into a coherent final output (using LLM)

```
         User Task
              │
              ▼
     ┌─────────────────┐
     │  1. DECOMPOSE    │──── LLM call: "Break this into subtasks"
     └────────┬────────┘
              │ subtask list
              ▼
     ┌─────────────────┐
     │  2. DELEGATE     │──── Registry lookup + worker execution
     └────────┬────────┘
              │ worker results
              ▼
     ┌─────────────────┐
     │  3. AGGREGATE    │──── LLM call: "Combine these results"
     └─────────────────┘
              │
              ▼
         Final Output
```

In [ ]:
class ManagerAgent:
    """Manager agent that decomposes tasks, delegates to workers, and aggregates results."""

    def __init__(self, name: str, registry: WorkerRegistry, max_retries: int = 2):
        self.name = name
        self.registry = registry
        self.max_retries = max_retries
        self.execution_log: List[Dict[str, Any]] = []

    def decompose(self, task: str, strategy: str = "flat") -> TaskDecomposition:
        """Decompose a complex task into subtasks using LLM."""
        if strategy == "flat":
            return self._flat_decompose(task)
        elif strategy == "hierarchical":
            return self._hierarchical_decompose(task)
        else:
            raise ValueError(f"Unknown strategy: {strategy}")

    def _flat_decompose(self, task: str) -> TaskDecomposition:
        """Break task into a flat list of independent subtasks."""
        prompt = f"""Break the following complex task into 3-5 independent subtasks.
For each subtask, provide:
1. A short ID (kebab-case, e.g., "research-solar")
2. A clear description of what needs to be done
3. Required skills (from: research, analysis, writing, editing, data_collection, comparison, evaluation, critical_thinking, communication, formatting)

Task: {task}

Respond in this exact JSON format:
{{"subtasks": [
    {{"id": "subtask-id", "description": "What to do", "skills": ["skill1", "skill2"]}},
    ...
]}}"""

        messages = [
            {"role": "system", "content": "You are a project manager who breaks complex tasks into clear, actionable subtasks. Always respond in valid JSON."},
            {"role": "user", "content": prompt},
        ]

        raw = generate(messages, max_new_tokens=500, temperature=0.4, do_sample=True)

        try:
            json_match = re.search(r'\{[\s\S]*\}', raw)
            if json_match:
                data = json.loads(json_match.group())
                subtasks = [
                    SubTask(
                        id=st["id"],
                        description=st["description"],
                        required_skills=st.get("skills", []),
                    )
                    for st in data.get("subtasks", [])
                ]
                return TaskDecomposition(task, subtasks, strategy="flat")
        except (json.JSONDecodeError, KeyError) as e:
            print(f"  ⚠ JSON parse error in decomposition: {e}")

        # Fallback: create generic subtasks
        return TaskDecomposition(task, [
            SubTask("research", "Research the topic thoroughly", ["research"]),
            SubTask("analyze", "Analyze the findings", ["analysis"]),
            SubTask("write", "Write the final output", ["writing"]),
        ], strategy="flat")

    def _hierarchical_decompose(self, task: str) -> TaskDecomposition:
        """Break task into subtasks that may have sub-subtasks (tree structure)."""
        prompt = f"""Break the following complex task into 2-3 HIGH-LEVEL phases.
Each phase should contain 2-3 specific subtasks.

Task: {task}

Respond in this exact JSON format:
{{"phases": [
    {{"phase": "phase-name", "subtasks": [
        {{"id": "subtask-id", "description": "What to do", "skills": ["skill1"]}}
    ]}}
]}}"""

        messages = [
            {"role": "system", "content": "You are a senior project manager who creates hierarchical work breakdown structures. Always respond in valid JSON."},
            {"role": "user", "content": prompt},
        ]

        raw = generate(messages, max_new_tokens=600, temperature=0.4, do_sample=True)

        all_subtasks = []
        try:
            json_match = re.search(r'\{[\s\S]*\}', raw)
            if json_match:
                data = json.loads(json_match.group())
                for phase in data.get("phases", []):
                    phase_name = phase.get("phase", "unknown")
                    for st in phase.get("subtasks", []):
                        subtask_id = f"{phase_name}-{st['id']}"
                        all_subtasks.append(SubTask(
                            id=subtask_id,
                            description=f"[{phase_name}] {st['description']}",
                            required_skills=st.get("skills", []),
                        ))
        except (json.JSONDecodeError, KeyError) as e:
            print(f"  ⚠ Hierarchical parse error: {e}")
            return self._flat_decompose(task)

        if not all_subtasks:
            return self._flat_decompose(task)
        return TaskDecomposition(task, all_subtasks, strategy="hierarchical")

    def delegate(self, decomposition: TaskDecomposition) -> TaskDecomposition:
        """Delegate each subtask to the best matching worker."""
        print(f"\n📤 Delegating {len(decomposition.subtasks)} subtasks...")

        for subtask in decomposition.subtasks:
            failed_workers = []
            success = False

            for attempt in range(self.max_retries):
                worker = self.registry.find_worker(
                    subtask.required_skills,
                    exclude=failed_workers,
                )
                if not worker:
                    subtask.status = TaskStatus.FAILED
                    subtask.error = "No matching worker found"
                    print(f"  ⚠ No worker for '{subtask.id}' (skills: {subtask.required_skills})")
                    break

                subtask = worker.execute(subtask)
                if subtask.status == TaskStatus.COMPLETED:
                    success = True
                    break
                else:
                    failed_workers.append(worker.name)
                    if attempt < self.max_retries - 1:
                        subtask.status = TaskStatus.REASSIGNED
                        print(f"    ↻ Reassigning '{subtask.id}' (attempt {attempt + 2})")

            self.execution_log.append({
                "subtask_id": subtask.id,
                "status": subtask.status.value,
                "worker": subtask.assigned_to,
                "attempts": subtask.attempts,
            })

        return decomposition

    def aggregate(self, task: str, decomposition: TaskDecomposition) -> str:
        """Aggregate worker results into a coherent final output."""
        completed = [st for st in decomposition.subtasks if st.status == TaskStatus.COMPLETED]
        failed = [st for st in decomposition.subtasks if st.status == TaskStatus.FAILED]

        results_text = ""
        for st in completed:
            results_text += f"\n--- Result from '{st.id}' (by {st.assigned_to}) ---\n"
            results_text += st.result + "\n"

        if failed:
            results_text += f"\n--- FAILED SUBTASKS ({len(failed)}) ---\n"
            for st in failed:
                results_text += f"- {st.id}: {st.error}\n"

        prompt = f"""You are assembling a final output from multiple worker results.

Original task: {task}

Worker results:
{results_text}

Your job:
1. Combine all worker results into a single, coherent document
2. Resolve any contradictions between worker outputs
3. Ensure smooth transitions between sections
4. Add an introduction and conclusion
5. If any subtasks failed, note what's missing

Produce the final, polished output."""

        messages = [
            {"role": "system", "content": "You are a senior editor who synthesizes multiple inputs into a cohesive document."},
            {"role": "user", "content": prompt},
        ]

        print("\n📥 Aggregating results...")
        return generate(messages, max_new_tokens=800, temperature=0.5)

    def run(self, task: str, strategy: str = "flat") -> Dict[str, Any]:
        """Execute the full decompose → delegate → aggregate cycle."""
        print(f"\n{'=' * 60}")
        print(f"🏢 Manager '{self.name}' received task")
        print(f"{'=' * 60}")
        start = time.time()

        # Step 1: Decompose
        print(f"\n📋 Step 1: Decomposing task (strategy: {strategy})...")
        decomposition = self.decompose(task, strategy)
        decomposition.show()

        # Step 2: Delegate
        print(f"\n👷 Step 2: Delegating to workers...")
        decomposition = self.delegate(decomposition)

        # Step 3: Aggregate
        print(f"\n📝 Step 3: Aggregating results...")
        final_output = self.aggregate(task, decomposition)

        elapsed = time.time() - start
        completed = sum(1 for st in decomposition.subtasks if st.status == TaskStatus.COMPLETED)
        total = len(decomposition.subtasks)

        print(f"\n{'=' * 60}")
        print(f"🏁 Task complete: {completed}/{total} subtasks succeeded ({elapsed:.1f}s)")
        print(f"{'=' * 60}")

        return {
            "final_output": final_output,
            "decomposition": decomposition,
            "execution_log": self.execution_log,
            "total_time": elapsed,
            "success_rate": completed / total if total > 0 else 0,
        }

print("✓ ManagerAgent ready")

## 6. Task Decomposition Strategies

### 6.1 — Flat vs. Hierarchical Decomposition

Let's see how the same complex task gets decomposed with each strategy.

In [ ]:
# Create a manager
manager = ManagerAgent(name="project_manager", registry=registry)

complex_task = (
    "Write a comprehensive report on renewable energy sources. "
    "Cover solar, wind, and hydroelectric power. For each source, "
    "discuss the technology, current costs, environmental impact, "
    "and future outlook. Include a comparison section and recommendations."
)

# Flat decomposition
print("━" * 60)
print("STRATEGY 1: Flat Decomposition")
print("━" * 60)
flat_decomp = manager.decompose(complex_task, strategy="flat")
flat_decomp.show()

# Hierarchical decomposition
print(f"\n{'━' * 60}")
print("STRATEGY 2: Hierarchical Decomposition")
print("━" * 60)
hier_decomp = manager.decompose(complex_task, strategy="hierarchical")
hier_decomp.show()

## 7. The Delegation Protocol

How the manager communicates with workers follows a clear protocol:

```
Manager                          Worker
  │                                │
  │  SubTask(id, description,      │
  │          required_skills)       │
  ├───────────────────────────────►│
  │                                │  Execute with specialized prompt
  │                                │  ...processing...
  │  SubTask(status=COMPLETED,     │
  │          result="...",         │
  │          assigned_to="name")   │
  │◄───────────────────────────────┤
  │                                │
  │  OR                            │
  │                                │
  │  SubTask(status=FAILED,        │
  │          error="...",          │
  │          attempts=N)           │
  │◄───────────────────────────────┤
```

**Key design choices:**
- Workers receive ONLY their subtask description — not the full original task
- Workers return structured SubTask objects — not raw strings
- The manager tracks attempts for retry logic
- Failed subtasks include error information for debugging

## 8. Example: Research Report Task

Let's run the full decompose → delegate → aggregate cycle on a real task.

In [ ]:
# Full manager-worker execution
report_result = manager.run(
    task="Write a report on the impact of artificial intelligence on healthcare. "
         "Cover diagnostic AI, drug discovery, patient care automation, "
         "ethical concerns, and future predictions.",
    strategy="flat",
)

print("\n" + "=" * 60)
print("FINAL OUTPUT")
print("=" * 60)
print(report_result["final_output"][:1500])
if len(report_result["final_output"]) > 1500:
    print(f"\n... ({len(report_result['final_output']) - 1500} more chars)")

print(f"\n📊 Execution Summary:")
print(f"  Success rate: {report_result['success_rate']:.0%}")
print(f"  Total time: {report_result['total_time']:.1f}s")
print(f"  Execution log:")
for entry in report_result["execution_log"]:
    print(f"    {entry['subtask_id']}: {entry['status']} (by {entry['worker']}, {entry['attempts']} attempt(s))")

### 8.1 — Same Task with Hierarchical Decomposition

Let's try the same task with hierarchical decomposition and compare.

In [ ]:
# Reset execution log
manager.execution_log = []

hier_result = manager.run(
    task="Write a report on the impact of artificial intelligence on healthcare. "
         "Cover diagnostic AI, drug discovery, patient care automation, "
         "ethical concerns, and future predictions.",
    strategy="hierarchical",
)

print("\n" + "=" * 60)
print("FINAL OUTPUT (Hierarchical)")
print("=" * 60)
print(hier_result["final_output"][:1500])
if len(hier_result["final_output"]) > 1500:
    print(f"\n... ({len(hier_result['final_output']) - 1500} more chars)")

## 9. Multi-Level Hierarchy — Managers Managing Managers

For very complex tasks, we can create **sub-managers** who each manage their own team of workers. The top-level manager delegates to sub-managers, who further delegate to workers.

```
              ┌──────────────────┐
              │  Senior Manager  │
              └────────┬─────────┘
                       │
          ┌────────────┼────────────┐
          ▼            ▼            ▼
     ┌─────────┐ ┌─────────┐ ┌─────────┐
     │ Research │ │Analysis │ │ Writing │
     │ Manager │ │ Manager │ │ Manager │
     └────┬────┘ └────┬────┘ └────┬────┘
       ┌──┼──┐     ┌──┼──┐     ┌──┼──┐
       ▼  ▼  ▼     ▼  ▼  ▼     ▼  ▼  ▼
      W1 W2 W3    W4 W5 W6    W7 W8 W9
```

In [ ]:
class SubManagerAgent:
    """A sub-manager that handles one phase of a larger project.

    Unlike ManagerAgent, SubManagerAgent focuses on a specific domain
    and has its own small team of workers.
    """

    def __init__(self, name: str, domain: str, workers: List[WorkerAgent]):
        self.name = name
        self.domain = domain
        self.workers = workers
        self.local_registry = WorkerRegistry()
        print(f"  Sub-manager '{name}' ({domain}):")
        for w in workers:
            self.local_registry.register(w)

    def handle_phase(self, phase_description: str) -> str:
        """Handle an entire phase by decomposing and delegating locally."""
        print(f"\n  📋 Sub-manager '{self.name}' handling: {phase_description[:60]}...")

        # Simple decomposition: split into 2 subtasks
        prompt = f"""Break this task into exactly 2 specific subtasks.
Task: {phase_description}

Respond in JSON: {{"subtasks": [{{"id": "id1", "description": "...", "skills": ["skill1"]}}, {{"id": "id2", "description": "...", "skills": ["skill1"]}}]}}"""

        messages = [
            {"role": "system", "content": f"You are a {self.domain} team lead. Respond in valid JSON."},
            {"role": "user", "content": prompt},
        ]

        raw = generate(messages, max_new_tokens=300, temperature=0.4, do_sample=True)
        subtasks = []
        try:
            json_match = re.search(r'\{[\s\S]*\}', raw)
            if json_match:
                data = json.loads(json_match.group())
                for st_data in data.get("subtasks", []):
                    subtasks.append(SubTask(
                        id=f"{self.name}-{st_data['id']}",
                        description=st_data["description"],
                        required_skills=st_data.get("skills", []),
                    ))
        except (json.JSONDecodeError, KeyError):
            subtasks = [SubTask(f"{self.name}-task1", phase_description, ["research"])]

        # Execute subtasks
        results = []
        for st in subtasks:
            worker = self.local_registry.find_worker(st.required_skills)
            if worker:
                st = worker.execute(st)
                if st.result:
                    results.append(st.result)
            else:
                # Fallback: use first available worker
                if self.workers:
                    st = self.workers[0].execute(st)
                    if st.result:
                        results.append(st.result)

        return "\n\n".join(results) if results else "No results produced."


class MultiLevelManager:
    """Top-level manager that delegates to sub-managers."""

    def __init__(self, name: str, sub_managers: List[SubManagerAgent]):
        self.name = name
        self.sub_managers = {sm.domain: sm for sm in sub_managers}

    def run(self, task: str) -> Dict[str, Any]:
        """Execute a multi-level hierarchical delegation."""
        print(f"\n{'=' * 60}")
        print(f"🏛 Senior Manager '{self.name}' — Multi-Level Execution")
        print(f"{'=' * 60}")
        start = time.time()

        # Decompose into phases (one per sub-manager domain)
        prompt = f"""Break this task into exactly {len(self.sub_managers)} phases,
one for each domain: {', '.join(self.sub_managers.keys())}.

Task: {task}

Respond in JSON: {{"phases": [{{"domain": "domain_name", "description": "phase description"}}]}}"""

        messages = [
            {"role": "system", "content": "You are a senior project director. Respond in valid JSON."},
            {"role": "user", "content": prompt},
        ]

        raw = generate(messages, max_new_tokens=400, temperature=0.4, do_sample=True)

        phases = {}
        try:
            json_match = re.search(r'\{[\s\S]*\}', raw)
            if json_match:
                data = json.loads(json_match.group())
                for phase in data.get("phases", []):
                    domain = phase.get("domain", "")
                    phases[domain] = phase.get("description", "")
        except (json.JSONDecodeError, KeyError):
            pass

        # Fallback: assign task to all sub-managers
        if not phases:
            for domain in self.sub_managers:
                phases[domain] = f"{domain} phase of: {task}"

        # Delegate to sub-managers
        phase_results = {}
        for domain, description in phases.items():
            sm = self.sub_managers.get(domain)
            if sm:
                phase_results[domain] = sm.handle_phase(description)
            else:
                print(f"  ⚠ No sub-manager for domain '{domain}'")

        # Aggregate all phase results
        combined = ""
        for domain, result in phase_results.items():
            combined += f"\n=== {domain.upper()} PHASE ===\n{result}\n"

        agg_prompt = f"""Combine these phase results into a cohesive final document.

Original task: {task}

Phase results:
{combined}

Create a well-structured, coherent final output with introduction and conclusion."""

        messages = [
            {"role": "system", "content": "You are a senior editor combining department outputs."},
            {"role": "user", "content": agg_prompt},
        ]

        final = generate(messages, max_new_tokens=800, temperature=0.5)
        elapsed = time.time() - start

        print(f"\n🏁 Multi-level execution complete ({elapsed:.1f}s)")
        return {"final_output": final, "phase_results": phase_results, "total_time": elapsed}

print("✓ Multi-level hierarchy classes ready")

### 9.1 — Building and Running a Multi-Level Hierarchy

In [ ]:
# Create specialized workers for each domain
research_team = [
    WorkerAgent("research_primary", ["research", "data_collection"],
                "You are a primary researcher. Gather detailed facts and data on the topic.", max_tokens=500),
    WorkerAgent("research_secondary", ["research", "information_gathering"],
                "You are a secondary researcher. Find additional perspectives and sources.", max_tokens=500),
]

analysis_team = [
    WorkerAgent("data_analyst", ["analysis", "evaluation", "comparison"],
                "You are a data analyst. Analyze findings, identify trends, compare options.", max_tokens=500),
    WorkerAgent("critical_thinker", ["analysis", "critical_thinking"],
                "You are a critical analyst. Evaluate arguments, find weaknesses, assess evidence.", max_tokens=500),
]

writing_team = [
    WorkerAgent("content_writer", ["writing", "communication"],
                "You are a content writer. Create clear, engaging prose from raw material.", max_tokens=500),
    WorkerAgent("editor_formatter", ["writing", "editing", "formatting"],
                "You are an editor. Polish text, fix errors, improve structure.", max_tokens=500),
]

# Create sub-managers
print("Creating sub-managers:")
research_mgr = SubManagerAgent("research_lead", "research", research_team)
analysis_mgr = SubManagerAgent("analysis_lead", "analysis", analysis_team)
writing_mgr = SubManagerAgent("writing_lead", "writing", writing_team)

# Create senior manager
senior = MultiLevelManager(
    "senior_director",
    [research_mgr, analysis_mgr, writing_mgr],
)

# Run a complex task
multi_result = senior.run(
    "Analyze the pros and cons of remote work vs. in-office work. "
    "Consider productivity, employee wellbeing, company culture, costs, and collaboration."
)

print("\n" + "=" * 60)
print("FINAL OUTPUT (Multi-Level Hierarchy)")
print("=" * 60)
print(multi_result["final_output"][:1200])
if len(multi_result["final_output"]) > 1200:
    print(f"\n... ({len(multi_result['final_output']) - 1200} more chars)")

## 10. Failure Handling — Robust Delegation

Real systems fail. Workers might produce garbage, time out, or crash. A good manager handles these gracefully:

1. **Retry** — try the same worker again (transient errors)
2. **Reassign** — try a different worker with matching skills
3. **Skip** — mark subtask as failed and continue with partial results
4. **Fallback** — manager does the subtask itself as last resort

In [ ]:
class RobustManagerAgent(ManagerAgent):
    """Manager with enhanced failure handling including fallback execution."""

    def __init__(self, name: str, registry: WorkerRegistry,
                 max_retries: int = 2, fallback_enabled: bool = True):
        super().__init__(name, registry, max_retries)
        self.fallback_enabled = fallback_enabled

    def delegate_with_fallback(self, decomposition: TaskDecomposition) -> TaskDecomposition:
        """Delegate with fallback: manager handles failed subtasks itself."""
        # First, try normal delegation
        decomposition = self.delegate(decomposition)

        if not self.fallback_enabled:
            return decomposition

        # Check for failures and handle them
        failed = [st for st in decomposition.subtasks if st.status == TaskStatus.FAILED]
        if failed:
            print(f"\n🔧 Manager fallback: handling {len(failed)} failed subtask(s)...")
            for subtask in failed:
                print(f"  ▶ Manager handling '{subtask.id}' directly...", end=" ")
                messages = [
                    {"role": "system", "content": "You are a versatile assistant. Complete the given task thoroughly."},
                    {"role": "user", "content": f"Task: {subtask.description}"},
                ]
                try:
                    start = time.time()
                    result = generate(messages, max_new_tokens=500, temperature=0.6)
                    subtask.result = result
                    subtask.status = TaskStatus.COMPLETED
                    subtask.assigned_to = f"{self.name}_fallback"
                    elapsed = time.time() - start
                    print(f"✓ ({elapsed:.1f}s)")
                except Exception as e:
                    subtask.error = f"Fallback also failed: {e}"
                    print(f"✗ ({e})")

        return decomposition

    def run(self, task: str, strategy: str = "flat") -> Dict[str, Any]:
        """Execute with enhanced failure handling."""
        print(f"\n{'=' * 60}")
        print(f"🏢 Robust Manager '{self.name}' received task")
        print(f"{'=' * 60}")
        start = time.time()

        print(f"\n📋 Decomposing (strategy: {strategy})...")
        decomposition = self.decompose(task, strategy)
        decomposition.show()

        print(f"\n👷 Delegating with fallback...")
        decomposition = self.delegate_with_fallback(decomposition)

        print(f"\n📝 Aggregating...")
        final = self.aggregate(task, decomposition)

        elapsed = time.time() - start
        completed = sum(1 for st in decomposition.subtasks if st.status == TaskStatus.COMPLETED)
        total = len(decomposition.subtasks)

        print(f"\n🏁 Done: {completed}/{total} succeeded ({elapsed:.1f}s)")
        return {
            "final_output": final,
            "decomposition": decomposition,
            "total_time": elapsed,
            "success_rate": completed / total if total > 0 else 0,
        }

# Test robust manager with our existing registry
robust_manager = RobustManagerAgent(
    name="robust_pm",
    registry=registry,
    max_retries=2,
    fallback_enabled=True,
)

robust_result = robust_manager.run(
    "Explain the causes and potential solutions for the global water crisis. "
    "Include data on water scarcity, pollution, and access inequality.",
    strategy="flat",
)

print(f"\n📊 Final success rate: {robust_result['success_rate']:.0%}")
print(f"Total time: {robust_result['total_time']:.1f}s")

## 11. Comparison: Flat vs. Manager-Worker vs. Multi-Level

Let's compare three approaches on the same complex task, measuring quality, cost (token count), and resilience.

In [ ]:
# Approach 1: Single flat agent
single_agent_prompt = """You are a comprehensive assistant. Complete the following task thoroughly.
Cover all aspects mentioned, organize clearly, and be detailed yet concise."""

def flat_approach(task: str) -> Dict[str, Any]:
    """Single agent does everything."""
    messages = [
        {"role": "system", "content": single_agent_prompt},
        {"role": "user", "content": task},
    ]
    start = time.time()
    result = generate(messages, max_new_tokens=800, temperature=0.6)
    return {"final_output": result, "total_time": time.time() - start, "approach": "flat"}

# The same test task for all approaches
comparison_task = (
    "Create a guide to starting a small business. Cover: "
    "1) Choosing a business idea, 2) Market research, "
    "3) Legal requirements, 4) Funding options, "
    "5) Marketing strategy. Make it practical and actionable."
)

print("━" * 60)
print("APPROACH 1: Single Agent (Flat)")
print("━" * 60)
flat_result = flat_approach(comparison_task)
print(f"Time: {flat_result['total_time']:.1f}s")
print(f"Output length: {len(flat_result['final_output'].split())} words")
print(flat_result["final_output"][:500])

print(f"\n{'━' * 60}")
print("APPROACH 2: Manager + Workers (Flat Delegation)")
print("━" * 60)
manager.execution_log = []
mw_result = manager.run(comparison_task, strategy="flat")
print(f"\nOutput preview:")
print(mw_result["final_output"][:500])

print(f"\n{'━' * 60}")
print("APPROACH 3: Multi-Level Hierarchy")
print("━" * 60)
ml_result = senior.run(comparison_task)
print(f"\nOutput preview:")
print(ml_result["final_output"][:500])

### 11.1 — Quality Comparison via LLM Judge

In [ ]:
def judge_quality(text: str, task: str) -> Dict[str, Any]:
    """Use LLM as judge to evaluate output quality."""
    prompt = f"""Rate this text on a task of: "{task[:100]}..."

Score 1-5 for each:
1. COMPLETENESS — Are all requested topics covered?
2. DEPTH — Is the coverage detailed and insightful?
3. ORGANIZATION — Is the text well-structured?
4. ACTIONABILITY — Are recommendations practical and specific?
5. COHERENCE — Does the text flow logically?

Respond in JSON: {{"completeness": N, "depth": N, "organization": N, "actionability": N, "coherence": N, "total": N, "comment": "..."}}

Text:
{text[:2000]}"""

    messages = [
        {"role": "system", "content": "You are an impartial quality judge. Respond in valid JSON only."},
        {"role": "user", "content": prompt},
    ]
    raw = generate(messages, max_new_tokens=200, temperature=0.2, do_sample=False)

    try:
        json_match = re.search(r'\{[\s\S]*\}', raw)
        if json_match:
            return json.loads(json_match.group())
    except (json.JSONDecodeError, AttributeError):
        pass
    return {"completeness": 3, "depth": 3, "organization": 3, "actionability": 3, "coherence": 3, "total": 15, "comment": "Parse error"}

print("Judging all three approaches...\n")

scores_flat = judge_quality(flat_result["final_output"], comparison_task)
scores_mw = judge_quality(mw_result["final_output"], comparison_task)
scores_ml = judge_quality(ml_result["final_output"], comparison_task)

# Comparison table
dims = ["completeness", "depth", "organization", "actionability", "coherence"]
print(f"{'Dimension':<16} {'Flat':>8} {'Mgr+Wkr':>8} {'Multi-Lv':>8}")
print("─" * 44)
for d in dims:
    print(f"{d:<16} {scores_flat.get(d, '?'):>8} {scores_mw.get(d, '?'):>8} {scores_ml.get(d, '?'):>8}")
print("─" * 44)
print(f"{'TOTAL':<16} {scores_flat.get('total', '?'):>8} {scores_mw.get('total', '?'):>8} {scores_ml.get('total', '?'):>8}")
print(f"{'Time (s)':<16} {flat_result['total_time']:>8.1f} {mw_result['total_time']:>8.1f} {ml_result['total_time']:>8.1f}")

print(f"\nComments:")
print(f"  Flat:      {scores_flat.get('comment', 'N/A')}")
print(f"  Mgr+Wkr:  {scores_mw.get('comment', 'N/A')}")
print(f"  Multi-Lv:  {scores_ml.get('comment', 'N/A')}")

## 12. When Hierarchy Hurts — Anti-Patterns

### 12.1 — Over-Decomposition

Breaking "Summarize this paragraph" into 5 subtasks with 3 levels of management is absurd. The overhead vastly exceeds the benefit.

**Rule of thumb**: If a single, well-prompted agent can produce acceptable output, skip the hierarchy.

### 12.2 — Lost Context Between Levels

Each delegation step **loses context**. The senior manager's understanding of the task is richer than what any sub-manager receives:

```
Senior Manager: "Write about renewable energy for policy makers" (knows full context)
     │
     ├──► Research Manager: "Research renewable energy" (lost: "for policy makers")
     │         │
     │         └──► Worker: "Find facts about solar" (lost: "renewable energy" framing)
```

### 12.3 — Coordination Overhead

| Levels | LLM Calls | Coordination Overhead |
|---|---|---|
| 1 (flat) | 1 | None |
| 2 (manager + workers) | N+2 (decompose + N workers + aggregate) | Moderate |
| 3 (multi-level) | N+M+3 (decompose at each level + workers + aggregate at each level) | High |

### 12.4 — Decision Framework

| Task Characteristics | Recommended Approach |
|---|---|
| Simple, single-focus task | Single agent |
| Multi-part task, clear subtasks | Manager + Workers |
| Large project, multiple domains | Multi-level hierarchy |
| Time-critical, needs low latency | Single agent or 2-level max |
| High reliability required | Manager + Workers with retry/fallback |

## 13. Delegation Patterns Reference

### Pattern 1: Broadcast Delegation
Manager sends the same task to multiple workers and picks the best result.

### Pattern 2: Scatter-Gather
Manager sends different subtasks to different workers, then gathers all results.

### Pattern 3: Pipeline + Hierarchy
Manager delegates to workers, then results flow through a pipeline for refinement.

### Pattern 4: Recursive Delegation
Sub-managers can further decompose and delegate, creating arbitrary depth.

In [ ]:
# Quick demo: Broadcast delegation (multiple workers, best result)
def broadcast_delegate(task: str, workers: List[WorkerAgent]) -> str:
    """Send same task to all workers, return the longest (most detailed) response."""
    print(f"📡 Broadcasting task to {len(workers)} workers...")
    results = []

    subtask = SubTask(id="broadcast", description=task, required_skills=[])
    for worker in workers:
        sub = SubTask(id=f"broadcast-{worker.name}", description=task)
        sub = worker.execute(sub)
        if sub.status == TaskStatus.COMPLETED:
            results.append((worker.name, sub.result))

    if not results:
        return "All workers failed."

    # Pick the longest response as "most detailed"
    best = max(results, key=lambda x: len(x[1]))
    print(f"\n  🏆 Best result from: {best[0]} ({len(best[1])} chars)")

    # Show comparison
    for name, result in results:
        marker = " ← SELECTED" if name == best[0] else ""
        print(f"  • {name}: {len(result)} chars, {len(result.split())} words{marker}")

    return best[1]

broadcast_result = broadcast_delegate(
    "What are the three most important skills for a software engineer in 2025?",
    [research_worker, analysis_worker, writing_worker],
)
print(f"\nBroadcast winner output:\n{broadcast_result[:400]}")

## 14. Design Guidelines for Hierarchical Systems

### Building Effective Hierarchies

1. **Keep decomposition coarse** — 3–5 subtasks, not 15. Each subtask should represent a meaningful unit of work.

2. **Skill tags should be specific enough for matching** — "research" and "analysis" are good; "do stuff" is not.

3. **Managers should add value** — The decomposition and aggregation steps should produce better results than simply concatenating subtask outputs. If a manager is just passing tasks through, remove that level.

4. **Plan for failure at every level** — Workers fail, sub-managers fail, even the top-level manager's decomposition can be poor. Build retry and fallback at each level.

5. **Log everything** — Record decomposition decisions, worker assignments, intermediate results, and timing. This is essential for debugging complex hierarchies.

6. **Test with simple tasks first** — Before using a hierarchy for complex work, verify each component works correctly in isolation.

## 15. Exercise: Build Your Own Hierarchy

Design a hierarchical agent system for one of these scenarios:

1. **Conference Talk Preparation**: Research Manager (topic research, audience analysis) + Content Manager (outline, slides, script) + Review Manager (fact-check, rehearsal feedback)

2. **Product Launch Plan**: Market Research Manager + Product Strategy Manager + Marketing Campaign Manager

3. **Code Project**: Architecture Manager + Implementation Manager + Testing Manager

Think about:
- What workers does each sub-manager need?
- What skills should workers have?
- Where might failures occur?
- What context is lost at each delegation level?

In [ ]:
# Exercise: Build a simple 2-level hierarchy for conference talk prep

# Create specialized workers for the task
topic_researcher = WorkerAgent(
    "topic_expert", ["research", "data_collection"],
    "You are a topic researcher. Find key facts, trends, and talking points about the topic. Be thorough.",
    max_tokens=500,
)

audience_analyst = WorkerAgent(
    "audience_expert", ["analysis", "evaluation"],
    "You are an audience analyst. Determine what the audience cares about, their knowledge level, and what would engage them.",
    max_tokens=400,
)

outline_writer = WorkerAgent(
    "outline_writer", ["writing", "communication"],
    "You are a presentation outline specialist. Create a clear, logical outline with sections, timing, and key messages.",
    max_tokens=500,
)

# Create a simple talk-prep manager using our existing infrastructure
talk_registry = WorkerRegistry()
print("Setting up Conference Talk Prep team:")
talk_registry.register(topic_researcher)
talk_registry.register(audience_analyst)
talk_registry.register(outline_writer)

talk_manager = RobustManagerAgent(
    name="talk_prep_manager",
    registry=talk_registry,
    max_retries=2,
    fallback_enabled=True,
)

talk_result = talk_manager.run(
    "Prepare a 15-minute conference talk on 'How Large Language Models Are Changing Software Development' "
    "for an audience of mid-career software engineers at a tech conference.",
    strategy="flat",
)

print("\n" + "=" * 60)
print("TALK PREPARATION OUTPUT")
print("=" * 60)
print(talk_result["final_output"][:1200])
if len(talk_result["final_output"]) > 1200:
    print(f"\n... ({len(talk_result['final_output']) - 1200} more chars)")

## 16. Summary — Key Takeaways

### What We Built
1. **SubTask & TaskDecomposition** — structured task representation with status tracking
2. **WorkerAgent** — specialized agent with typed skills and capability scoring
3. **WorkerRegistry** — service discovery for workers by skill matching
4. **ManagerAgent** — decompose → delegate → aggregate orchestration
5. **SubManagerAgent & MultiLevelManager** — recursive hierarchical delegation
6. **RobustManagerAgent** — enhanced failure handling with retry, reassign, and fallback

### Core Principles

| Principle | Why It Matters |
|---|---|
| **Decomposition** | Complex tasks are easier to solve as focused subtasks |
| **Skill matching** | Right worker for the right job improves quality |
| **Failure isolation** | One worker's failure doesn't corrupt the whole result |
| **Aggregation** | Combining specialized outputs creates coherent wholes |
| **Observability** | Logging at every level enables debugging complex systems |

### Comparison of Approaches

| Approach | Best For | LLM Calls | Quality | Resilience |
|---|---|---|---|---|
| **Single Agent** | Simple tasks | 1 | Good for easy tasks | None |
| **Manager + Workers** | Multi-part tasks | N+2 | Better coverage | Retry/reassign |
| **Multi-Level** | Complex projects | Many | Best for complex work | Cascading fallback |

### Connection to Previous and Next Notebooks
- **Notebook 19 (Pipelines)** gave us sequential chains — great for step-by-step refinement
- **This notebook** adds hierarchical decomposition — great for divide-and-conquer
- Together, these patterns cover most multi-agent architectures in practice

---
*Notebook 20 of the Castalia Agents course.*

## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the agents/ directory